# Run Experiment

In [12]:
import sys, json
from pathlib import Path
import pandas as pd
sys.path.insert(0, "..")
import config
from src.single_agent import run_single_agent
from src.multi_agent import run_multi_agent, MultiAgentPrompts
from evaluation.evaluator import evaluate

config.RAW_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
config.AGGREGATED_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Setup complete")

Setup complete


In [13]:
import yaml

def load_prompt(path):
    return (Path('..') / 'prompts' / path).read_text(encoding='utf-8')

_raw = yaml.safe_load((Path('..') / 'evaluation' / 'tasks.yaml').read_text(encoding='utf-8'))

_cp_prompts = MultiAgentPrompts(
    orchestrator=load_prompt('context_pollution/multi_agent/orchestrator.md'),
    reader=load_prompt('context_pollution/multi_agent/subagent_reader.md'),
)
_ver_prompts = MultiAgentPrompts(
    orchestrator=load_prompt('verification/multi_agent/orchestrator.md'),
    executor=load_prompt('verification/multi_agent/subagent_executor.md'),
    verifier=load_prompt('verification/multi_agent/subagent_verifier.md'),
)
_par_prompts_base = {
    'orchestrator': load_prompt('parallelization/multi_agent/orchestrator.md'),
    'researcher': load_prompt('parallelization/multi_agent/subagent_researcher.md'),
}

_MAS_PROMPTS = {
    config.SCENARIO_CONTEXT_POLLUTION: _cp_prompts,
    config.SCENARIO_VERIFICATION: _ver_prompts,
}
_SAS_PROMPT = {
    config.SCENARIO_CONTEXT_POLLUTION: load_prompt('context_pollution/single_agent.md'),
    config.SCENARIO_PARALLELIZATION: load_prompt('parallelization/single_agent.md'),
    config.SCENARIO_VERIFICATION: load_prompt('verification/single_agent.md'),
}

TASKS = []
for t in _raw['tasks']:
    scenario = t['scenario']
    if scenario == config.SCENARIO_PARALLELIZATION:
        mas_prompts = MultiAgentPrompts(**_par_prompts_base, companies=t['companies'])
    else:
        mas_prompts = _MAS_PROMPTS[scenario]

    TASKS.append({
        'scenario': scenario,
        'variant': t['variant'],
        'question': t['question'].strip(),
        'reference': t['ground_truth'],
        'sas_prompt': _SAS_PROMPT[scenario],
        'mas_prompts': mas_prompts,
    })

print(f"Total tasks:      {len(TASKS)}")
print(f"Total agent runs: {len(TASKS) * 2 * config.N_RUNS}")

Total tasks:      9
Total agent runs: 90


In [14]:
all_records = []

for task in TASKS:
    scenario = task["scenario"]
    variant = task["variant"]
    print(f"\n{'=' * 60}")
    print(f"Scenario: {scenario} | Variant: {variant}")
    print(f"{'=' * 60}")

    for run_idx in range(1, config.N_RUNS + 1):
        for architecture in ["single_agent", "multi_agent"]:
            print(f"  run {run_idx}/{config.N_RUNS} | {architecture} ... ", end="", flush=True)

            if architecture == "single_agent":
                agent_result = await run_single_agent(
                    scenario=scenario,
                    system_prompt=task["sas_prompt"],
                    question=task["question"],
                )
            else:
                agent_result = await run_multi_agent(
                    scenario=scenario,
                    prompts=task["mas_prompts"],
                    question=task["question"],
                )

            eval_result = evaluate(
                scenario=scenario,
                architecture=architecture,
                reference_answer=task["reference"],
                agent_answer=agent_result.answer,
            )

            print(f"quality={eval_result.quality_score}  ({eval_result.reason})")

            record = {
                "scenario":      scenario,
                "variant":       variant,
                "architecture":  architecture,
                "run":           run_idx,
                "answer":        agent_result.answer,
                "quality_score": eval_result.quality_score,
                "reason":        eval_result.reason,
                "agent_error":   agent_result.error,
                "eval_error":    eval_result.error,
            }
            all_records.append(record)

            raw_path = config.RAW_RESULTS_DIR / f"{scenario}__{variant}__{architecture}__run{run_idx}.json"
            raw_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"\nDone. {len(all_records)} runs completed.")


Scenario: context_pollution | Variant: apple
  run 1/5 | single_agent ... quality=1.0  (All values match)
  run 1/5 | multi_agent ... quality=1.0  (All values match)
  run 2/5 | single_agent ... quality=1.0  (All values match)
  run 2/5 | multi_agent ... quality=1.0  (All values match)
  run 3/5 | single_agent ... quality=1.0  (All values match)
  run 3/5 | multi_agent ... quality=1.0  (All values match)
  run 4/5 | single_agent ... quality=1.0  (All values match)
  run 4/5 | multi_agent ... quality=1.0  (All values match)
  run 5/5 | single_agent ... quality=1.0  (All values match)
  run 5/5 | multi_agent ... quality=1.0  (All values match)

Scenario: context_pollution | Variant: microsoft
  run 1/5 | single_agent ... quality=1.0  (All values match)
  run 1/5 | multi_agent ... quality=1.0  (All values match)
  run 2/5 | single_agent ... quality=1.0  (All values match)
  run 2/5 | multi_agent ... quality=1.0  (All values match)
  run 3/5 | single_agent ... quality=1.0  (All values mat

TenK falling back to legacy parser for 'part_ii_item_2' (filing: 0001045810-26-000021). New parser sections available: ['part_i_item_1', 'part_i_item_1a', 'part_i_item_1b', 'part_iv_item_16', 'part_iv_signatures', 'part_i_item_1c', 'part_i_item_2', 'part_i_item_3', 'part_i_item_4', 'part_ii_item_5', 'part_ii_item_6', 'part_ii_item_7', 'part_ii_item_7a', 'part_ii_item_8', 'part_ii_item_9', 'part_ii_item_9a', 'part_ii_item_9b', 'part_ii_item_9c', 'part_iii_item_10', 'part_iii_item_11', 'part_iii_item_12', 'part_iii_item_13', 'part_iii_item_14', 'part_iv_item_15']. This fallback will be removed in v6.0.


quality=1.0  (All values match)
  run 1/5 | multi_agent ... quality=1.0  (All values match)
  run 2/5 | single_agent ... quality=1.0  (All values match)
  run 2/5 | multi_agent ... quality=1.0  (All values match)
  run 3/5 | single_agent ... quality=1.0  (All values match)
  run 3/5 | multi_agent ... quality=1.0  (All values match)
  run 4/5 | single_agent ... quality=1.0  (All values match)
  run 4/5 | multi_agent ... quality=1.0  (All values match)
  run 5/5 | single_agent ... quality=1.0  (All values match)
  run 5/5 | multi_agent ... quality=1.0  (All values match)

Scenario: parallelization | Variant: rnd
  run 1/5 | single_agent ... quality=1.0  (All values match)
  run 1/5 | multi_agent ... quality=1.0  (All values match)
  run 2/5 | single_agent ... 

TenK falling back to legacy parser for 'part_iv_item_8' (filing: 0001045810-26-000021). New parser sections available: ['part_i_item_1', 'part_i_item_1a', 'part_i_item_1b', 'part_iv_item_16', 'part_iv_signatures', 'part_i_item_1c', 'part_i_item_2', 'part_i_item_3', 'part_i_item_4', 'part_ii_item_5', 'part_ii_item_6', 'part_ii_item_7', 'part_ii_item_7a', 'part_ii_item_8', 'part_ii_item_9', 'part_ii_item_9a', 'part_ii_item_9b', 'part_ii_item_9c', 'part_iii_item_10', 'part_iii_item_11', 'part_iii_item_12', 'part_iii_item_13', 'part_iii_item_14', 'part_iv_item_15']. This fallback will be removed in v6.0.


quality=1.0  (All values match)
  run 2/5 | multi_agent ... quality=1.0  (All values match)
  run 3/5 | single_agent ... quality=1.0  (All values match)
  run 3/5 | multi_agent ... quality=1.0  (All values match)
  run 4/5 | single_agent ... quality=1.0  (All values match)
  run 4/5 | multi_agent ... quality=1.0  (All values match)
  run 5/5 | single_agent ... quality=1.0  (All values match)
  run 5/5 | multi_agent ... quality=1.0  (All values match)

Scenario: parallelization | Variant: total_revenue
  run 1/5 | single_agent ... quality=1.0  (All values match)
  run 1/5 | multi_agent ... quality=1.0  (All values match)
  run 2/5 | single_agent ... quality=1.0  (All values match)
  run 2/5 | multi_agent ... quality=1.0  (All values match)
  run 3/5 | single_agent ... quality=1.0  (All values match)
  run 3/5 | multi_agent ... quality=1.0  (All values match)
  run 4/5 | single_agent ... quality=1.0  (All values match)
  run 4/5 | multi_agent ... quality=1.0  (All values match)
  run 5/5

In [15]:
df = pd.DataFrame([
    {
        "scenario":      r["scenario"],
        "variant":       r["variant"],
        "architecture":  r["architecture"],
        "run":           r["run"],
        "quality_score": r["quality_score"],
        "agent_error":   r["agent_error"],
        "eval_error":    r["eval_error"],
    }
    for r in all_records
])

agg = df.groupby(["scenario", "variant", "architecture"]).agg(
    quality_mean=("quality_score", "mean"),
    quality_std=("quality_score", "std"),
    n_runs=("run", "count"),
).round(3).reset_index()

out_raw = config.AGGREGATED_RESULTS_DIR / "results.csv"
out_agg = config.AGGREGATED_RESULTS_DIR / "results_aggregated.csv"
df.to_csv(out_raw, index=False)
agg.to_csv(out_agg, index=False)
print(f"Saved: {out_raw}")
print(f"Saved: {out_agg}\n")
agg

Saved: /Users/dmytrosvyrydenko/Documents/Projects/single-vs-multi-agent-llm/notebooks/../results/aggregated/results.csv
Saved: /Users/dmytrosvyrydenko/Documents/Projects/single-vs-multi-agent-llm/notebooks/../results/aggregated/results_aggregated.csv



,scenario,variant,architecture,quality_mean,quality_std,n_runs
0,context_pollution,apple,multi_agent,1.0,0.0,5
1,context_pollution,apple,single_agent,1.0,0.0,5
2,context_pollution,microsoft,multi_agent,1.0,0.0,5
3,context_pollution,microsoft,single_agent,1.0,0.0,5
4,context_pollution,nvidia,multi_agent,1.0,0.0,5
5,context_pollution,nvidia,single_agent,1.0,0.0,5
6,parallelization,data_center,multi_agent,1.0,0.0,5
7,parallelization,data_center,single_agent,1.0,0.0,5
8,parallelization,rnd,multi_agent,1.0,0.0,5
9,parallelization,rnd,single_agent,1.0,0.0,5
